![image.png](https://i.imgur.com/4fN73lZ.png)

## Setup

We install with **uv** and use `imageio` for GIF rendering (consistent with the other RL labs).

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
# Bootstraps uv via pip, then installs into the current kernel/venv. If you already
# run from the project's .venv (created with `uv sync`), this is essentially a no-op.
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" gymnasium flappy_bird_gymnasium imageio matplotlib torch

# Content

In this lab we build a **GRPO** (Group Relative Policy Optimization) agent from scratch in PyTorch and train it on the Flappy Bird game.

For the environment we use `flappy_bird_gymnasium`. You can read more about it [here](https://github.com/markub3327/flappy-bird-gymnasium).

> This is the companion to the **PPO** Flappy Bird lab. GRPO is deliberately built on the *same* PPO-Clip objective so you can see exactly what changes — **the critic is gone**.

## GRPO

> **Exercise:** This is the student version. Complete the three tasks in the `GRPOAgent`: `TASK 1` (the group-relative advantage in `group_advantages`), and `TASK 2` (the probability ratio) and `TASK 3` (the clipped surrogate objective) in `grpo_loss`. Unfinished tasks raise `NotImplementedError`. Hints give the *formula*, not the code. A fully worked version is in `Day-4_GRPO_Flappy_Bird_Solution.ipynb`.

## GRPO: PPO without a critic

Recall the Actor-Critic loop: the actor learns a policy and the **critic** $V_\phi(s)$ learns a value baseline. Training two networks at once is the main source of instability, and the critic is roughly half the memory.

**Group Relative Policy Optimization** (GRPO; Shao et al., 2024) throws the critic away. Instead of asking a value network "how good was this state?", it asks a much simpler question: *how good was this whole run compared to a batch of sibling runs from the same policy?*

Concretely, per "prompt/state" it samples a **group** of $G$ outputs under the frozen policy $\pi_{\theta_\text{old}}$ and scores each with a reward $R_i$. The advantage is just the reward centred on the **group mean** — a Monte-Carlo baseline, no value function:

$$\hat{A}_i = \frac{R_i - \overbrace{\mathrm{mean}(R_1,\dots,R_G)}^{\text{the ``group relative'' baseline}}}{\underbrace{\mathrm{std}(R_1,\dots,R_G)}_{\text{optional per-group normaliser}}}$$

Two things to keep straight (straight from the lecture):

- The **mean-subtraction** *is* GRPO — it is the critic-free baseline replacing $V_\phi$.
- The **std-division** is a *separate, optional* stabiliser (the same per-batch advantage normalisation used in PPO), applied per group. A Monte-Carlo baseline does not *need* it.

That single scalar $\hat{A}_i$ is then shared by **every timestep of output $i$** and plugged into the **exact same PPO-Clip objective** you already implemented.

### Mapping GRPO onto a game

GRPO was designed for LLMs, where a "prompt" yields $G$ sampled completions each earning one scalar reward. We map that onto Flappy Bird the natural way:

| GRPO / RLHF term | Here (Flappy Bird) |
| --- | --- |
| prompt / state | the reset environment |
| one output / completion | one **full episode** |
| reward $R_i$ of the output | the **episode return** (its score) |
| a group of $G$ outputs | $G$ episodes rolled from the same policy |

So one *update* = collect $G$ episodes, score each by its return, center on the group mean, and take PPO-Clip steps. This is the same shape as the **RLHF** pipeline you'll revisit on **Day 9** — a preference-model reward per completion, group-normalised, fed to PPO-Clip.

> **A caveat worth knowing.** Critic-free GRPO leans on episodes *terminating* to separate good runs from bad ones (Flappy ends the moment the bird crashes, so this holds). But every timestep in an episode shares one advantage, so credit assignment is coarse — expect GRPO here to learn a solid flap, just less sample-efficiently than the critic-based PPO version.

### The KL-to-reference term (optional here)

The full GRPO objective adds one more piece: a **KL penalty to a fixed reference policy**, $\beta\,\mathrm{KL}(\pi_\theta \,\|\, \pi_\text{ref})$. In **RLHF** that reference is the pre-RL (SFT) model, and the KL is what stops the policy drifting into gibberish that games the learned reward model.

In a from-scratch control task there is no meaningful "SFT model" to anchor to, and PPO-Clip already keeps each step conservative — so we leave the KL **off by default** (`beta_kl = 0.0`). We still snapshot the initial policy as a reference so you can switch it on:

> **Try it:** set `beta_kl` to a small value (e.g. `0.01`) and re-run. Watch how it slows the policy's drift from where it started. It is not needed to learn Flappy, but wiring it up is exactly the mechanism that matters on Day 9.

In [ ]:
import copy
import gymnasium as gym
import flappy_bird_gymnasium
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

# These tiny networks are env-step bound, so CPU is actually faster than GPU here.
device = torch.device("cpu")
print("device:", device)

## Let's define our model

The only architectural change from the PPO lab is the **removed critic head** — GRPO gets its baseline from the group, not from a value network. Everything else (the body, the actor head) is identical.

In [ ]:
# Policy network: the PPO ActorCritic body with the actor head ONLY.
# There is no critic head -- that is the whole point of GRPO.
class PolicyNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 32)
        self.actor = nn.Linear(32, output_dim)   # action logits (no value head)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.actor(x)

In [ ]:
class GRPOAgent:
    def __init__(self, input_dim, output_dim, lr=1e-3, epsilon=0.2, k_epochs=8,
                 entropy_coef=0.01, beta_kl=0.0, device="cpu"):
        self.policy = PolicyNet(input_dim, output_dim).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.epsilon = epsilon
        self.k_epochs = k_epochs
        self.entropy_coef = entropy_coef
        self.beta_kl = beta_kl            # weight of the optional KL-to-reference term (0 = off)
        self.reference = None             # frozen reference policy, set by snapshot_reference()
        self.device = device

    def snapshot_reference(self):
        """Freeze a copy of the current policy as the KL reference (see the note above)."""
        self.reference = copy.deepcopy(self.policy)
        for p in self.reference.parameters():
            p.requires_grad_(False)

    def select_action(self, state):
        """Sample an action under the current policy; return (action, log_prob).
        The log_prob is kept as the OLD log-prob for the PPO ratio."""
        state = torch.tensor(np.asarray(state), dtype=torch.float32, device=self.device)
        with torch.no_grad():
            dist = Categorical(logits=self.policy(state))
            action = dist.sample()
        return action.item(), dist.log_prob(action)

    def group_advantages(self, episode_returns):
        """The GRPO advantage: one scalar per episode."""
        R = torch.tensor(episode_returns, dtype=torch.float32, device=self.device)
        # TASK 1: the group-relative advantage (this IS GRPO, replacing the critic).
        # HINT: centre R on the GROUP MEAN, then divide by the group std to normalise:
        #       (R - R.mean()) / (R.std() + 1e-8).
        raise NotImplementedError("TASK 1: compute the group-relative advantage")

    def grpo_loss(self, states, actions, advantages, old_log_probs):
        """The PPO-Clip loss -- with NO value term (there is no critic).
        Optionally adds a KL penalty to the frozen reference policy."""
        dist = Categorical(logits=self.policy(states))
        log_probs = dist.log_prob(actions)

        # TASK 2: the probability ratio r_t = pi_new(a|s) / pi_old(a|s).
        # HINT: a ratio of probabilities is the exp of a DIFFERENCE of log-probs
        #       (log_probs - old_log_probs).
        ratio = None

        # TASK 3: the PPO clipped surrogate (the policy loss).
        # HINT: surr1 = ratio * advantages; surr2 = clamp(ratio, 1-epsilon, 1+epsilon) * advantages;
        #       policy loss = -min(surr1, surr2).mean().
        policy_loss = None

        if ratio is None or policy_loss is None:
            raise NotImplementedError("Complete TASK 2 and TASK 3 in grpo_loss")

        entropy = dist.entropy().mean()
        loss = policy_loss - self.entropy_coef * entropy

        if self.beta_kl > 0 and self.reference is not None:      # optional; off by default
            with torch.no_grad():
                ref_log_probs = Categorical(logits=self.reference(states)).log_prob(actions)
            kl = (log_probs - ref_log_probs).mean()
            loss = loss + self.beta_kl * kl

        return loss

    def update(self, states, actions, advantages, old_log_probs):
        """Reuse the group for k_epochs of gradient steps on the GRPO loss."""
        states = torch.tensor(np.array(states), dtype=torch.float32, device=self.device)
        actions = torch.tensor(actions, dtype=torch.int64, device=self.device)
        old_log_probs = torch.stack(old_log_probs).detach().to(self.device)
        advantages = advantages.detach().to(self.device)
        for _ in range(self.k_epochs):
            loss = self.grpo_loss(states, actions, advantages, old_log_probs)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

## Collecting a group

One *update* consumes a **group** of $G$ episodes rolled out under the current (frozen) policy. We flatten every transition into one big batch, and remember each episode's **return** (its score) and **length** so we can broadcast that episode's single advantage back across its own timesteps.

In [ ]:
def collect_group(env, agent, group_size, max_steps):
    """Roll out `group_size` episodes under the current policy.
    Returns the flattened transitions plus the per-episode returns GRPO compares."""
    states, actions, old_log_probs = [], [], []
    episode_returns, episode_lengths = [], []
    for _ in range(group_size):
        state = env.reset()[0]
        ep_return, ep_len = 0.0, 0
        for _ in range(max_steps):
            action, log_prob = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            states.append(state)
            actions.append(action)
            old_log_probs.append(log_prob)
            ep_return += reward
            ep_len += 1
            state = next_state
            if terminated or truncated:
                break
        episode_returns.append(ep_return)
        episode_lengths.append(ep_len)
    return states, actions, old_log_probs, episode_returns, episode_lengths

## Initialize the environment and model

In [ ]:
env = gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=False)

input_dim = env.observation_space.shape[0]
output_dim = env.action_space.n

group_size = 8        # G: episodes per update (small groups work best -- see the lecture / paper)
num_updates = 1500    # one update == one group of G episodes
max_steps = 1000

lr = 1e-3
epsilon = 0.2
k_epochs = 8
entropy_coef = 0.01
beta_kl = 0.0         # optional KL-to-reference weight; 0.0 = off (see the note above)

agent = GRPOAgent(input_dim, output_dim, lr, epsilon, k_epochs,
                  entropy_coef, beta_kl, device=device)

## Training the model

Each update: collect a group, turn the $G$ returns into $G$ advantages, **broadcast** each episode's advantage across its own timesteps, and take PPO-Clip steps.

In [ ]:
scores = []
agent.snapshot_reference()   # freeze the KL reference (inert while beta_kl == 0)

for update in range(num_updates):
    states, actions, old_log_probs, episode_returns, episode_lengths = \
        collect_group(env, agent, group_size, max_steps)

    # one scalar advantage per episode ...
    advantages = agent.group_advantages(episode_returns)
    # ... broadcast across that episode's own timesteps, aligning with the flat batch
    adv_per_step = torch.cat([advantages[i].repeat(n) for i, n in enumerate(episode_lengths)])

    agent.update(states, actions, adv_per_step, old_log_probs)

    scores.append(np.mean(episode_returns))
    if (update + 1) % 25 == 0:
        print(f"Update {update + 1:5d} | avg return over group (last 25 updates): "
              f"{np.mean(scores[-25:]):7.2f}")

env.close()

## Reward curve

In [ ]:
window = 25
smoothed = np.convolve(scores, np.ones(window) / window, mode="valid")
plt.figure(figsize=(9, 4))
plt.plot(smoothed)
plt.xlabel("update")
plt.ylabel(f"group-mean return ({window}-update moving avg)")
plt.title("GRPO on Flappy Bird")
plt.grid(alpha=0.3)
plt.show()

## Visualizing model's performance

In [ ]:
# Roll out the trained policy and save it as a GIF (Gymnasium-native)
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")  # headless rendering (e.g. Colab)
import imageio.v2 as imageio
from IPython.display import Image, display

os.makedirs("video", exist_ok=True)
eval_env = gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=False)
state = eval_env.reset()[0]
frames = []
for _ in range(2000):
    frames.append(eval_env.render())
    action, _ = agent.select_action(state)
    state, reward, terminated, truncated, _ = eval_env.step(action)
    if terminated or truncated:
        break
eval_env.close()
imageio.mimsave("video/flappy_grpo.gif", frames, fps=24, loop=0)

In [ ]:
display(Image(filename="video/flappy_grpo.gif"))